In [5]:
import pandas as pd
df = pd.DataFrame()
df = pd.read_csv(r'data/diabetic_data.csv')

In [6]:
df.dtypes

encounter_id                 int64
patient_nbr                  int64
race                        object
gender                      object
age                         object
weight                      object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
payer_code                  object
medical_specialty           object
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nateglinide                 object
chlorpropamide              object
glimepiride         

In [7]:
"""

1. encounter_id - unique identifier of an encounter (guess: should be removed)
2. patient_nbr - unique identifier of a patient (guess: should be removed)
3. race
4. gender
5. age
6. weight
7. admission_type_id - 9 distinct admission types (integers 1-9)
8. discharge_disposition_id - 26 distinct discharge types (integers 1-26)
9. admission_source_id - 17 distinct admission sources (integers 1-17)
10. time_in_hospital - in days
11. payer_code - insurance or self-pay (hints at patient's financial situation and if they would be able to afford to stay admitted for longer)
12. medical_specialty - of physician (84 types)
13. num_lab_procedures
14. num_procedures - other than lab tests
15. num_medications - number of distinct medications administered
17. number_emergency - emergency visits in the year leading up to admission
18. number_inpatient - inpatient visits in the year leading up to admission
19. diag_1 - primary diagnosis (848 values)
20. diag_2 - secondary diagnosis (923 values)
21. diag_3 - additional seconday diagnosis (954 values)
22. number_diagnoses - number of diagnoses in system (i.e. how many conditions the patient has)
23. max_glu_serum - blood glucose level (categorical: >200, >300, normal, or none)
24. A1Cresult - normal is close to 7% in healthy adults (categorical: >7, >8, normal, or none)

the following are all medications with 4 categories about its dosage (up, down, steady, or none):
25. metformin
26. repaglinide
27. nateglinide
28. chlorpropamide
29. glimepiride
30. acetohexamide
31. glipizide
32. glyburide
33. tolbutamide
34. pioglitazone
35. rosiglitazone
36. acarbose
37. miglitol
38. troglitazone
39. tolazamide
40. examide
41. citoglipton
42. insulin
43. glyburide-metformin
44. glipizide-metformin
45. glimepiride-pioglitazone
46. metformin-rosiglitazone
47. metformin-pioglitazone


48. change - if there was a change in type or dosage of diabetes medication (binary: no or yes)
49. diabetesMed - if medication was prescribed at discharge
50. readmitted - categorical: <30, >30, or no
"""

"\n\n1. encounter_id - unique identifier of an encounter (guess: should be removed)\n2. patient_nbr - unique identifier of a patient (guess: should be removed)\n3. race\n4. gender\n5. age\n6. weight\n7. admission_type_id - 9 distinct admission types (integers 1-9)\n8. discharge_disposition_id - 26 distinct discharge types (integers 1-26)\n9. admission_source_id - 17 distinct admission sources (integers 1-17)\n10. time_in_hospital - in days\n11. payer_code - insurance or self-pay (hints at patient's financial situation and if they would be able to afford to stay admitted for longer)\n12. medical_specialty - of physician (84 types)\n13. num_lab_procedures\n14. num_procedures - other than lab tests\n15. num_medications - number of distinct medications administered\n17. number_emergency - emergency visits in the year leading up to admission\n18. number_inpatient - inpatient visits in the year leading up to admission\n19. diag_1 - primary diagnosis (848 values)\n20. diag_2 - secondary diagn

In [8]:
import sqlite3
import os

connection = sqlite3.connect('/data/diabetic_data.db')
df.to_sql('diabetic_data', connection, if_exists='replace', index=False)
connection.close()

print("Saved to:", os.path.abspath('/data/diabetic_data.db'))

OperationalError: unable to open database file

In [ ]:
new_connection = sqlite3.connect('data/diabetic_data.db')
query = "SELECT COUNT(*), payer_code FROM diabetic_data WHERE readmitted = '<30' GROUP BY payer_code"
result = pd.read_sql_query(query, new_connection)
print(result)
new_connection.close()

    COUNT(*) payer_code
0       4627          ?
1        426         BC
2         13         CH
3        198         CM
4        214         CP
5         64         DM
6        644         HM
7       3810         MC
8        416         MD
9          9         MP
10       136         OG
11         7         OT
12        44         PO
13         7         SI
14       510         SP
15       227         UN
16         5         WC


In [ ]:
import pandas as pd
import sqlite3

# your UCI patient data - already loaded as df from previous days
# confirming it's still in memory
print("UCI data shape:", df.shape)

# load the CMS hospital general information
hospital_info = pd.read_csv(
    r'/data/hospitals_current_data/Hospital_General_Information.csv'
)

# load the CMS readmission reduction program data
# this file has hospital-level readmission penalty information
readm_by_hospital = pd.read_csv(
    r'\data\hospitals_current_data\FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv'
)

print("Hospital general info shape:", hospital_info.shape)
print("Readmission program shape:", readm_by_hospital.shape)

# readmission program shape is large due to the fact that 5 readmission types are measured for each hospital

UCI data shape: (101766, 50)
Hospital general info shape: (5432, 38)
Readmission program shape: (18330, 12)


In [ ]:
# we only want columns that are meaningful for enrichment
# dropping footnote columns, address details, phone numbers etc.
hosp_clean = hospital_info[[
    'Facility ID',
    'Facility Name', 
    'State',
    'Hospital Type',
    'Hospital Ownership',
    'Hospital overall rating'
]].copy()

# rename to lowercase with underscores - consistent with your UCI column style
hosp_clean.columns = [
    'facility_id',
    'facility_name',
    'state',
    'hospital_type',
    'hospital_ownership',
    'overall_rating'
]

# 'Hospital overall rating' contains strings like 'Not Available'
# pd.to_numeric with errors='coerce' turns those into NaN instead of crashing
hosp_clean['overall_rating'] = pd.to_numeric(
    hosp_clean['overall_rating'], errors='coerce'
)

print(hosp_clean.dtypes)
print("\nMissing values:\n", hosp_clean.isnull().sum())

facility_id            object
facility_name          object
state                  object
hospital_type          object
hospital_ownership     object
overall_rating        float64
dtype: object

Missing values:
 facility_id              0
facility_name            0
state                    0
hospital_type            0
hospital_ownership       0
overall_rating        2250
dtype: int64


In [ ]:
# ── 2. CLEAN HOSPITAL GENERAL INFO ───────────────────────────────────────────

hosp_clean = hospital_info[[
    'Facility ID',
    'Facility Name',
    'State',
    'Hospital Type',
    'Hospital Ownership',
    'Hospital overall rating'
]].copy()

hosp_clean.columns = [
    'facility_id',
    'facility_name',
    'state',
    'hospital_type',
    'hospital_ownership',
    'overall_rating'
]

# convert rating to numeric - 'Not Available' becomes NaN
hosp_clean['overall_rating'] = pd.to_numeric(
    hosp_clean['overall_rating'], errors='coerce'
)

# standardize facility ID to 6-digit string with leading zeros
# e.g. 10001 becomes 010001
hosp_clean['facility_id'] = hosp_clean['facility_id'].astype(str).str.zfill(6)

print("\nHospital general info cleaned:")
print(hosp_clean.dtypes)
print("Missing ratings:", hosp_clean['overall_rating'].isnull().sum())


# ── 3. CLEAN READMISSION PROGRAM FILE ────────────────────────────────────────

# convert numeric columns - 'Too Few to Report' becomes NaN
readm_by_hospital['Excess Readmission Ratio'] = pd.to_numeric(
    readm_by_hospital['Excess Readmission Ratio'], errors='coerce'
)
readm_by_hospital['Predicted Readmission Rate'] = pd.to_numeric(
    readm_by_hospital['Predicted Readmission Rate'], errors='coerce'
)
readm_by_hospital['Expected Readmission Rate'] = pd.to_numeric(
    readm_by_hospital['Expected Readmission Rate'], errors='coerce'
)
readm_by_hospital['Number of Discharges'] = pd.to_numeric(
    readm_by_hospital['Number of Discharges'], errors='coerce'
)

# standardize facility ID to match hosp_clean
readm_by_hospital['Facility ID'] = readm_by_hospital['Facility ID'].astype(str).str.zfill(6)


# ── 4. COLLAPSE READMISSION FILE TO ONE ROW PER HOSPITAL ─────────────────────

# all six conditions - general hospital quality summary
all_conditions_summary = readm_by_hospital.groupby(['Facility ID', 'State']).agg(
    avg_excess_readm_ratio   = ('Excess Readmission Ratio',   'mean'),
    avg_predicted_readm_rate = ('Predicted Readmission Rate', 'mean'),
    avg_expected_readm_rate  = ('Expected Readmission Rate',  'mean'),
    num_conditions_measured  = ('Measure Name',               'count')
).reset_index().round(4)

# heart failure and COPD only - diabetes-relevant proxy
# diabetic patients have significantly higher comorbidity rates for both
diabetes_relevant = readm_by_hospital[
    readm_by_hospital['Measure Name'].isin(['READM-30-HF-HRRP', 'READM-30-COPD-HRRP'])
]

diabetic_proxy_summary = diabetes_relevant.groupby(['Facility ID', 'State']).agg(
    avg_excess_readm_hf_copd    = ('Excess Readmission Ratio',   'mean'),
    avg_predicted_readm_hf_copd = ('Predicted Readmission Rate', 'mean'),
    avg_expected_readm_hf_copd  = ('Expected Readmission Rate',  'mean')
).reset_index().round(4)

print("\nAll conditions summary shape:    ", all_conditions_summary.shape)
print("Diabetic proxy summary shape:    ", diabetic_proxy_summary.shape)


# ── 5. JOIN ALL THREE TABLES INTO ONE ENRICHED HOSPITAL TABLE ────────────────

# first join: hospital general info + all conditions summary
hosp_enriched = hosp_clean.merge(
    all_conditions_summary,
    left_on='facility_id',
    right_on='Facility ID',
    how='left'
).drop(columns=['Facility ID', 'State'])

hosp_enriched = hosp_enriched.rename(columns={'State_x': 'state'})

# second join: add the diabetes-relevant proxy columns
hosp_enriched = hosp_enriched.merge(
    diabetic_proxy_summary[['Facility ID',
                             'avg_excess_readm_hf_copd',
                             'avg_predicted_readm_hf_copd',
                             'avg_expected_readm_hf_copd']],
    left_on='facility_id',
    right_on='Facility ID',
    how='left'
).drop(columns=['Facility ID'])

print("\nFinal enriched hospital table shape:", hosp_enriched.shape)
print(hosp_enriched.head(3))


# ── 6. COMPUTE STATE-LEVEL AVERAGES ──────────────────────────────────────────

state_quality = hosp_enriched.groupby('state').agg(
    num_hospitals            = ('facility_id',              'count'),
    avg_star_rating          = ('overall_rating',           'mean'),
    avg_excess_readm_ratio   = ('avg_excess_readm_ratio',   'mean'),
    avg_excess_readm_hf_copd = ('avg_excess_readm_hf_copd', 'mean')
).reset_index().round(2)

print("\nState quality table shape:", state_quality.shape)
print(state_quality.sort_values('avg_star_rating', ascending=False).head(5))


# ── 7. COMPUTE NATIONAL BENCHMARKS ───────────────────────────────────────────

national_benchmark = pd.DataFrame([{
    'total_hospitals_rated':         int(hosp_enriched['overall_rating'].count()),
    'national_avg_star_rating':      round(hosp_enriched['overall_rating'].mean(), 2),
    'national_avg_excess_readm':     round(hosp_enriched['avg_excess_readm_ratio'].mean(), 4),
    'national_avg_predicted_readm':  round(hosp_enriched['avg_predicted_readm_rate'].mean(), 2),
    'national_avg_excess_hf_copd':   round(hosp_enriched['avg_excess_readm_hf_copd'].mean(), 4),
    'pct_hospitals_below_expected':  round(
        (hosp_enriched['avg_excess_readm_ratio'] < 1.0).sum() /
         hosp_enriched['avg_excess_readm_ratio'].count() * 100, 1
    )
}])

print("\nNational Benchmarks:")
print(national_benchmark.T)


# ── 8. SAVE ALL TABLES TO SQLITE ─────────────────────────────────────────────

os.makedirs('data', exist_ok=True)
connection = sqlite3.connect('data/diabetic_data.db')

df.to_sql(
    'diabetic_data', connection, if_exists='replace', index=False
)
hosp_enriched.to_sql(
    'hospital_quality', connection, if_exists='replace', index=False
)
state_quality.to_sql(
    'state_quality', connection, if_exists='replace', index=False
)
national_benchmark.to_sql(
    'national_benchmark', connection, if_exists='replace', index=False
)

connection.close()
print("\nAll four tables saved to data/diabetic_data.db")


# ── 9. VERIFY EVERYTHING LANDED CORRECTLY ────────────────────────────────────

connection = sqlite3.connect('data/diabetic_data.db')

audit = pd.read_sql("""
    SELECT 'diabetic_data'      AS table_name, COUNT(*) AS rows FROM diabetic_data
    UNION ALL
    SELECT 'hospital_quality'   AS table_name, COUNT(*) AS rows FROM hospital_quality
    UNION ALL
    SELECT 'state_quality'      AS table_name, COUNT(*) AS rows FROM state_quality
    UNION ALL
    SELECT 'national_benchmark' AS table_name, COUNT(*) AS rows FROM national_benchmark
""", connection)

connection.close()
print("\nDatabase audit:")
print(audit)


Hospital general info cleaned:
facility_id            object
facility_name          object
state                  object
hospital_type          object
hospital_ownership     object
overall_rating        float64
dtype: object
Missing ratings: 2250

All conditions summary shape:     (3055, 6)
Diabetic proxy summary shape:     (3055, 5)

Final enriched hospital table shape: (5432, 13)
  facility_id                    facility_name state         hospital_type  \
0      010001  SOUTHEAST HEALTH MEDICAL CENTER    AL  Acute Care Hospitals   
1      010005         MARSHALL MEDICAL CENTERS    AL  Acute Care Hospitals   
2      010006     NORTH ALABAMA MEDICAL CENTER    AL  Acute Care Hospitals   

                            hospital_ownership  overall_rating  \
0  Government - Hospital District or Authority             4.0   
1  Government - Hospital District or Authority             3.0   
2                                  Proprietary             2.0   

   avg_excess_readm_ratio  avg_predi